# nnUNet — Complete Colab Pro Notebook

Full pipeline: install → preprocess → train → predict → evaluate.

**Before running:** Runtime → Change runtime type → **GPU** (T4 or better)

Uses source patching (not subclassing) to avoid nnUNet's `locals()` bug.

## Step 0 — Install nnUNet

In [ ]:
!pip install nnunetv2 nibabel -q
print('✓ nnUNet v2 installed')

In [ ]:
# Verify GPU is available
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ NO GPU — go to Runtime → Change runtime type → GPU')

## Step 1 — Download Dataset

In [ ]:
!curl -L -o images.npz https://github.com/DivyanshuTak/Ultrasoud_Unet_Segmentation/raw/refs/heads/main/images.npz
!curl -L -o masks.npz https://github.com/DivyanshuTak/Ultrasoud_Unet_Segmentation/raw/refs/heads/main/masks.npz
print('✓ Dataset downloaded')

## Step 2 — Preprocess & Convert to nnUNet Format

Same preprocessing as your INIA pipeline. Same seed, same split.

In [ ]:
import numpy as np
import nibabel as nib
import os

# ── Load ──
images = np.load('images.npz')['images']
masks = np.load('masks.npz')['masks']

# ── Preprocess (identical to INIA) ──
images = images[:, 24:]                    # crop top 24 rows
masks = masks[:, 24:]
images = images[:, :, :, 0]                # grayscale
images = images / 255.0                    # normalize
masks = (masks > 0).astype(np.float32)     # binarize

# Pad to 320x320
def pad(x):
    return np.pad(x[..., None], ((0, 0), (22, 22), (10, 10), (0, 0)))

images = pad(images)
masks = pad(masks)

# Shuffle (same seed=42 as INIA for fair comparison)
rng = np.random.RandomState(42)
idx = rng.permutation(len(images))
images = images[idx]
masks = masks[idx]

# Split: 180 train / 28 test (same as INIA)
X_train, y_train = images[:180], masks[:180]
X_test, y_test = images[180:], masks[180:]

print(f'Train: {X_train.shape}')
print(f'Test:  {X_test.shape}')

## Step 3 — Set Up nnUNet Folder Structure & Convert to NIfTI

In [ ]:
# ── Folder structure ──
DATASET_ID = 101
DATASET_NAME = f'Dataset{DATASET_ID:03d}_CardiacUS'

BASE = '/content/nnUNet'
RAW_DIR = os.path.join(BASE, 'nnUNet_raw', DATASET_NAME)
PREPROCESSED_DIR = os.path.join(BASE, 'nnUNet_preprocessed')
RESULTS_DIR = os.path.join(BASE, 'nnUNet_results')
TEST_LABELS_DIR = os.path.join(BASE, 'test_labels')
PREDICTIONS_DIR = os.path.join(BASE, 'predictions')

for d in [
    os.path.join(RAW_DIR, 'imagesTr'),
    os.path.join(RAW_DIR, 'labelsTr'),
    os.path.join(RAW_DIR, 'imagesTs'),
    PREPROCESSED_DIR,
    RESULTS_DIR,
    TEST_LABELS_DIR,
    PREDICTIONS_DIR,
]:
    os.makedirs(d, exist_ok=True)

# ── Environment variables (nnUNet reads these) ──
os.environ['nnUNet_raw'] = os.path.join(BASE, 'nnUNet_raw')
os.environ['nnUNet_preprocessed'] = PREPROCESSED_DIR
os.environ['nnUNet_results'] = RESULTS_DIR

print(f'✓ Folders created')
print(f'  Raw:          {RAW_DIR}')
print(f'  Preprocessed: {PREPROCESSED_DIR}')
print(f'  Results:      {RESULTS_DIR}')

In [ ]:
# ── Convert numpy → NIfTI ──
def save_nifti(arr_2d, path):
    """Save 2D array as 3D NIfTI with trivial z=1 dimension."""
    vol = arr_2d.squeeze()[np.newaxis, :, :]
    nib.save(nib.Nifti1Image(vol.astype(np.float32), np.eye(4)), path)

# Training images + labels
for i in range(len(X_train)):
    cid = f'case_{i:04d}'
    save_nifti(X_train[i], os.path.join(RAW_DIR, 'imagesTr', f'{cid}_0000.nii.gz'))
    save_nifti(y_train[i], os.path.join(RAW_DIR, 'labelsTr', f'{cid}.nii.gz'))

# Test images
for i in range(len(X_test)):
    cid = f'case_{i+180:04d}'
    save_nifti(X_test[i], os.path.join(RAW_DIR, 'imagesTs', f'{cid}_0000.nii.gz'))

# Test ground truth (for evaluation later)
for i in range(len(y_test)):
    cid = f'case_{i+180:04d}'
    save_nifti(y_test[i], os.path.join(TEST_LABELS_DIR, f'{cid}.nii.gz'))

print(f'✓ Saved {len(X_train)} training + {len(X_test)} test NIfTI files')

In [ ]:
# ── Generate dataset.json ──
from nnunetv2.dataset_conversion.generate_dataset_json import generate_dataset_json

generate_dataset_json(
    RAW_DIR,
    channel_names={0: 'ultrasound'},
    labels={'background': 0, 'ventricle': 1},
    num_training_cases=len(X_train),
    file_ending='.nii.gz'
)
print('✓ dataset.json created')

## Step 4 — nnUNet Plan & Preprocess

Automatic configuration: nnUNet analyzes your data and decides on architecture, patch size, batch size, etc.

In [ ]:
!nnUNetv2_plan_and_preprocess -d 101 --verify_dataset_integrity -c 2d
print('\n✓ Planning and preprocessing complete')

## Step 5 — Patch nnUNet to 50 Epochs & Train

**Why patching instead of subclassing?** nnUNet's `__init__` uses a `locals()` trick that breaks
any custom trainer subclass. Patching the source directly is the only reliable workaround.

Training: **2D config, fold 0, 50 epochs.** Takes ~30-60 min on T4 GPU.

In [ ]:
# ── Patch nnUNet source: 1000 epochs → 50 epochs ──
import nnunetv2.training.nnUNetTrainer.nnUNetTrainer as trainer_mod
import shutil

source_path = trainer_mod.__file__

# Backup original source
backup_path = source_path + '.bak'
if not os.path.exists(backup_path):
    shutil.copy(source_path, backup_path)
    print(f'✓ Backed up original to {backup_path}')

# Read, patch, write
with open(source_path, 'r') as f:
    content = f.read()

if 'self.num_epochs = 1000' in content:
    content = content.replace('self.num_epochs = 1000', 'self.num_epochs = 50')
    with open(source_path, 'w') as f:
        f.write(content)
    print('✓ Patched: 1000 epochs → 50 epochs')
elif 'self.num_epochs = 50' in content:
    print('✓ Already patched to 50 epochs')
else:
    print('⚠️ Could not find epoch setting — check nnUNet version')

In [ ]:
# ── Verify the patch worked ──
with open(source_path, 'r') as f:
    for line in f:
        if 'num_epochs' in line and 'self.num_epochs' in line:
            print(f'Verified: {line.strip()}')
            break

In [ ]:
# ══════════════════════════════════════════
# TRAIN — this is the big one (~30-60 min)
# ══════════════════════════════════════════
# Using DEFAULT trainer (no -tr flag) since we patched the source
!nnUNetv2_train 101 2d 0

## Step 6 — Verify Training Completed

Check that the results folder was created.

In [ ]:
# Check that training created output files
import os

results_base = '/content/nnUNet/nnUNet_results/Dataset101_CardiacUS'
expected_dir = os.path.join(results_base, 'nnUNetTrainer__nnUNetPlans__2d', 'fold_0')

if os.path.exists(expected_dir):
    contents = os.listdir(expected_dir)
    print(f'✓ Training output folder exists with {len(contents)} files:')
    for f in sorted(contents)[:10]:
        print(f'  {f}')
    if len(contents) > 10:
        print(f'  ... and {len(contents) - 10} more')
else:
    print('❌ Training folder NOT found. Training may have failed.')
    print(f'Expected: {expected_dir}')
    # List what IS in results
    if os.path.exists(results_base):
        print(f'\nContents of {results_base}:')
        for item in os.listdir(results_base):
            print(f'  {item}')

## Step 7 — Run Predictions on Test Set

In [ ]:
# No -tr flag since we used the default trainer
!nnUNetv2_predict \
    -i /content/nnUNet/nnUNet_raw/Dataset101_CardiacUS/imagesTs \
    -o /content/nnUNet/predictions \
    -d 101 -c 2d -f 0

# Verify predictions exist
import glob
pred_files = glob.glob('/content/nnUNet/predictions/*.nii.gz')
print(f'\n✓ Generated {len(pred_files)} predictions')

## Step 8 — Evaluate: Compute Dice & IoU

In [ ]:
import numpy as np
import nibabel as nib
import glob

def compute_dice(pred, gt, smooth=1e-6):
    pred, gt = pred.flatten(), gt.flatten()
    intersection = np.sum(pred * gt)
    return (2.0 * intersection + smooth) / (np.sum(pred) + np.sum(gt) + smooth)

def compute_iou(pred, gt, smooth=1e-6):
    pred, gt = pred.flatten(), gt.flatten()
    intersection = np.sum(pred * gt)
    union = np.sum(pred) + np.sum(gt) - intersection
    return (intersection + smooth) / (union + smooth)

pred_files = sorted(glob.glob('/content/nnUNet/predictions/*.nii.gz'))
gt_files = sorted(glob.glob('/content/nnUNet/test_labels/*.nii.gz'))

print(f'Predictions: {len(pred_files)}')
print(f'Ground truth: {len(gt_files)}')

dice_scores, iou_scores = [], []
for p, g in zip(pred_files, gt_files):
    pred = (nib.load(p).get_fdata() > 0.5).astype(np.float32)
    gt = (nib.load(g).get_fdata() > 0.5).astype(np.float32)
    dice_scores.append(compute_dice(pred, gt))
    iou_scores.append(compute_iou(pred, gt))

mean_dice = np.mean(dice_scores)
mean_iou = np.mean(iou_scores)

print(f'\n╔══════════════════════════════════════╗')
print(f'║   nnUNet Results (50 ep, fold 0, 2d) ║')
print(f'╠══════════════════════════════════════╣')
print(f'║   Mean Dice: {mean_dice:.4f}                 ║')
print(f'║   Mean IoU:  {mean_iou:.4f}                 ║')
print(f'║   Samples:   {len(dice_scores):>4d}                   ║')
print(f'╚══════════════════════════════════════╝')

## Step 9 — Full 4-Architecture Comparison

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

comparison = {
    'UNet':    {'Dice': 0.9527, 'IoU': 0.9096},
    'UNet++':  {'Dice': 0.9513, 'IoU': 0.9072},
    'UNet3++': {'Dice': 0.0664, 'IoU': None},
    'nnUNet':  {'Dice': round(mean_dice, 4), 'IoU': round(mean_iou, 4)},
}

df = pd.DataFrame(comparison).T
df.index.name = 'Architecture'

print('\n========== FINAL 4-ARCHITECTURE COMPARISON ==========')
print(df.to_string())
print('======================================================')

# ── Bar Charts ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']
archs = df.index.tolist()
x = np.arange(len(archs))

for ax, col, title in [(axes[0], 'Dice', 'Test Dice by Architecture'),
                        (axes[1], 'IoU', 'Test IoU by Architecture')]:
    vals = df[col].fillna(0)
    bars = ax.bar(x, vals, 0.5, color=colors)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel(col)
    ax.set_xticks(x)
    ax.set_xticklabels(archs)
    ax.set_ylim(0, 1.08)
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(vals):
        if v > 0:
            ax.text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Chart saved as comparison_chart.png')

## Step 10 — Save & Download Results

In [ ]:
# Save comparison table
df.to_csv('final_4_architecture_comparison.csv')
print('✓ Saved final_4_architecture_comparison.csv')

# Download
from google.colab import files
files.download('final_4_architecture_comparison.csv')
files.download('comparison_chart.png')

## (Optional) Restore nnUNet to Original 1000 Epochs

In [ ]:
# Run this if you want to undo the patch
# import shutil
# source_path = '/usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py'
# backup_path = source_path + '.bak'
# if os.path.exists(backup_path):
#     shutil.copy(backup_path, source_path)
#     print('✓ Restored original nnUNet source')
# else:
#     print('No backup found')